**Step 1 | Install and initialize PySpark**

In [2]:
!pip -q install pyspark

In [3]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark import StorageLevel
import time
spark = SparkSession.builder \
    .appName("PySparkPractice") \
    .getOrCreate()
products_data = [
    ("P01", "Laptop", "Electronics", 60000),
    ("P02", "Monitor", "Electronics", 14000),
    ("P03", "Headphones", "Accessories", 3500),
    ("P04", "Printer", "Office", 17000),
    ("P05", "Router", "Networking", 4500),
    ("P06", "Tablet", "Electronics", 24000)
]
products_df = spark.createDataFrame(
    products_data,
    ["product_id", "product_name", "category", "list_price"]
)
stores_data = [
    ("S01", "Hyderabad", "South"),
    ("S02", "Bengaluru", "South"),
    ("S03", "Mumbai", "West"),
    ("S04", "Delhi", "North")
]
stores_df = spark.createDataFrame(
    stores_data,
    ["store_id", "city", "region"]
)
products_df.show()
stores_df.show()
sales_df = (
    spark.range(1, 100001)
    .withColumnRenamed("id", "order_id")
    .withColumn(
        "order_date",
        F.date_add(
            F.to_date(F.lit("2026-01-01")),
            (F.col("order_id") % 180).cast("int")
        )
    )
    .withColumn(
        "product_id",
        F.concat(
            F.lit("P"),
            F.lpad(
                ((F.col("order_id") % 6) + 1).cast("string"),
                2, "0"
            )
        )
    )
    .withColumn(
        "store_id",
        F.concat(
            F.lit("S"),
            F.lpad(
                ((F.floor(F.col("order_id") / 3) % 4) + 1)
                .cast("string"),
                2, "0"
            )
        )
    )
    .withColumn(
        "quantity",
        ((F.col("order_id") % 8) + 1).cast("int")
    )
    .withColumn(
        "discount_pct",
        F.when(
            F.col("order_id") % 19 == 0,
            F.lit(None).cast("int")
        ).otherwise(
            ((F.floor(F.col("order_id") / 5) % 4) * 5)
            .cast("int")
        )
    )
    .withColumn(
        "order_status",
        F.when(F.col("order_id") % 11 == 0, "Cancelled")
         .when(F.col("order_id") % 7 == 0, "Returned")
         .otherwise("Completed")
    )

    .withColumn(
        "payment_method",
        F.when(F.col("order_id") % 3 == 0, "UPI")
         .when(F.col("order_id") % 3 == 1, "Card")
         .otherwise("Cash")
    )

    .withColumn(
        "salesperson_id",
        F.when(
            F.col("order_id") % 17 == 0,
            F.lit(None).cast("string")
        ).otherwise(
            F.concat(
                F.lit("EMP"),
                F.lpad(
                    ((F.col("order_id") % 12) + 1).cast("string"),
                    2, "0"
                )
            )
        )
    )

    .join(
        products_df.select(
            "product_id",
            F.col("list_price").alias("unit_price")
        ),
        "product_id",
        "left"
    )

    .select(
        "order_id", "order_date", "store_id",
        "product_id", "quantity", "unit_price",
        "discount_pct", "order_status",
        "payment_method", "salesperson_id"
    )
)

print("Total Records:", sales_df.count())
sales_df.printSchema()
sales_df.show(10, truncate=False)














+----------+------------+-----------+----------+
|product_id|product_name|   category|list_price|
+----------+------------+-----------+----------+
|       P01|      Laptop|Electronics|     60000|
|       P02|     Monitor|Electronics|     14000|
|       P03|  Headphones|Accessories|      3500|
|       P04|     Printer|     Office|     17000|
|       P05|      Router| Networking|      4500|
|       P06|      Tablet|Electronics|     24000|
+----------+------------+-----------+----------+

+--------+---------+------+
|store_id|     city|region|
+--------+---------+------+
|     S01|Hyderabad| South|
|     S02|Bengaluru| South|
|     S03|   Mumbai|  West|
|     S04|    Delhi| North|
+--------+---------+------+

Total Records: 100000
root
 |-- order_id: long (nullable = false)
 |-- order_date: date (nullable = true)
 |-- store_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: long (nullable = true)
 |-- discount_p

**Part A | DataFrame transformations**

In [4]:
#1
sales_df.select(
    'order_id',
    'product_id',
    'quantity',
    'unit_price'
).show()

+--------+----------+--------+----------+
|order_id|product_id|quantity|unit_price|
+--------+----------+--------+----------+
|       3|       P04|       4|     17000|
|       9|       P04|       2|     17000|
|      15|       P04|       8|     17000|
|      21|       P04|       6|     17000|
|       2|       P03|       3|      3500|
|       8|       P03|       1|      3500|
|      14|       P03|       7|      3500|
|      20|       P03|       5|      3500|
|       4|       P05|       5|      4500|
|      10|       P05|       3|      4500|
|      16|       P05|       1|      4500|
|       5|       P06|       6|     24000|
|      11|       P06|       4|     24000|
|      17|       P06|       2|     24000|
|       1|       P02|       2|     14000|
|       7|       P02|       8|     14000|
|      13|       P02|       6|     14000|
|      19|       P02|       4|     14000|
|       6|       P01|       7|     60000|
|      12|       P01|       5|     60000|
+--------+----------+--------+----

In [5]:
#2
from pyspark.sql.functions import col
filtered = sales_df.filter(
    "order_status == 'Completed' and quantity > 5"
)
filtered.show()

+--------+----------+--------+----------+--------+----------+------------+------------+--------------+--------------+
|order_id|order_date|store_id|product_id|quantity|unit_price|discount_pct|order_status|payment_method|salesperson_id|
+--------+----------+--------+----------+--------+----------+------------+------------+--------------+--------------+
|      15|2026-01-16|     S02|       P04|       8|     17000|          15|   Completed|           UPI|         EMP04|
|      39|2026-02-09|     S02|       P04|       8|     17000|          15|   Completed|           UPI|         EMP04|
|      45|2026-02-15|     S04|       P04|       6|     17000|           5|   Completed|           UPI|         EMP10|
|      69|2026-03-11|     S04|       P04|       6|     17000|           5|   Completed|           UPI|         EMP10|
|      38|2026-02-08|     S01|       P03|       7|      3500|        NULL|   Completed|          Cash|         EMP03|
|      62|2026-03-04|     S01|       P03|       7|      

In [6]:
#3
sales_df = sales_df.withColumn(
    'gross_revenue',
    col('quantity') * col('unit_price')
)
sales_df.show()


+--------+----------+--------+----------+--------+----------+------------+------------+--------------+--------------+-------------+
|order_id|order_date|store_id|product_id|quantity|unit_price|discount_pct|order_status|payment_method|salesperson_id|gross_revenue|
+--------+----------+--------+----------+--------+----------+------------+------------+--------------+--------------+-------------+
|       3|2026-01-04|     S02|       P04|       4|     17000|           0|   Completed|           UPI|         EMP04|        68000|
|       9|2026-01-10|     S04|       P04|       2|     17000|           5|   Completed|           UPI|         EMP10|        34000|
|      15|2026-01-16|     S02|       P04|       8|     17000|          15|   Completed|           UPI|         EMP04|       136000|
|      21|2026-01-22|     S04|       P04|       6|     17000|           0|    Returned|           UPI|         EMP10|       102000|
|       2|2026-01-03|     S01|       P03|       3|      3500|           0|  

In [7]:
#4
sales_df = sales_df.withColumnRenamed(
    'order_status',
    'transaction_status'
)
sales_df = sales_df.drop('payment_method')
sales_df.show()


+--------+----------+--------+----------+--------+----------+------------+------------------+--------------+-------------+
|order_id|order_date|store_id|product_id|quantity|unit_price|discount_pct|transaction_status|salesperson_id|gross_revenue|
+--------+----------+--------+----------+--------+----------+------------+------------------+--------------+-------------+
|       3|2026-01-04|     S02|       P04|       4|     17000|           0|         Completed|         EMP04|        68000|
|       9|2026-01-10|     S04|       P04|       2|     17000|           5|         Completed|         EMP10|        34000|
|      15|2026-01-16|     S02|       P04|       8|     17000|          15|         Completed|         EMP04|       136000|
|      21|2026-01-22|     S04|       P04|       6|     17000|           0|          Returned|         EMP10|       102000|
|       2|2026-01-03|     S01|       P03|       3|      3500|           0|         Completed|         EMP03|        10500|
|       8|2026-0

**Part B | Aggregations and joins**

In [8]:
#5
from pyspark.sql.functions import sum
sales_df.groupBy('product_id').agg(
    sum('quantity').alias('Total quantity sold')
).orderBy('product_id').show()

+----------+-------------------+
|product_id|Total quantity sold|
+----------+-------------------+
|       P01|              66668|
|       P02|              83336|
|       P03|              66667|
|       P04|              83334|
|       P05|              66665|
|       P06|              83330|
+----------+-------------------+



In [9]:
#6
joined_df = sales_df.join(
    products_df,
    on = 'product_id',
    how = 'inner'
)
joined_df.groupby('category').agg(
    sum('gross_revenue').alias('Total gross revenue')
).show()



+-----------+-------------------+
|   category|Total gross revenue|
+-----------+-------------------+
|     Office|         1416678000|
| Networking|          299992500|
|Electronics|         7166704000|
|Accessories|          233334500|
+-----------+-------------------+



In [10]:
#7
from pyspark.sql.functions import count
joined_df = sales_df.join(
    stores_df,
    on = 'store_id',
    how = 'inner'
)
joined_df.groupby('city').agg(
    count('order_id').alias('Total orders')
).show()


+---------+------------+
|     city|Total orders|
+---------+------------+
|   Mumbai|       24999|
|    Delhi|       24999|
|Bengaluru|       25001|
|Hyderabad|       25001|
+---------+------------+



In [11]:
#8
all_df = sales_df.join(
    products_df,
    on = 'product_id',
    how = 'inner'
).join(
    stores_df,
    on = 'store_id',
    how = 'inner'
)
all_df.filter(
    col('transaction_status') == 'Completed'
).groupby('product_id','product_name').agg(
    sum('gross_revenue').alias('Total gross revenue')
).orderBy('Total gross revenue',ascending=False).show(5)

+----------+------------+-------------------+
|product_id|product_name|Total gross revenue|
+----------+------------+-------------------+
|       P01|      Laptop|         3116940000|
|       P06|      Tablet|         1558416000|
|       P04|     Printer|         1103912000|
|       P02|     Monitor|          909076000|
|       P05|      Router|          233766000|
+----------+------------+-------------------+
only showing top 5 rows


**Part C | Null handling and conditions**

In [12]:
#9
print("Null values in Discount_pct:", sales_df.filter(
    col('discount_pct').isNull()
).count())
print("Null values in salesperson_id:", sales_df.filter(
    col('salesperson_id').isNull()
).count())


Null values in Discount_pct: 5263
Null values in salesperson_id: 5882


In [13]:
#10
sales_df1 = sales_df.na.fill(0,subset=['discount_pct'])
print("Null values in Discount_pct:", sales_df1.filter(
    col('discount_pct').isNull()
).count())

Null values in Discount_pct: 0


In [14]:
#11
print("Total number of records: ", sales_df1.count())
sales_df1 = sales_df1.na.drop(subset=['salesperson_id'])
print("After removing records:",sales_df1.count())

Total number of records:  100000
After removing records: 94118


In [15]:
#12
from pyspark.sql.functions import when
sales_df = sales_df.withColumn(
    'order_size',
    when((col('quantity') >= 1) & (col('quantity') <= 3), 'Small')
    .when((col('quantity') >= 4) & (col('quantity') <= 6), 'Medium')
    .otherwise('Large')
)
sales_df.show()

+--------+----------+--------+----------+--------+----------+------------+------------------+--------------+-------------+----------+
|order_id|order_date|store_id|product_id|quantity|unit_price|discount_pct|transaction_status|salesperson_id|gross_revenue|order_size|
+--------+----------+--------+----------+--------+----------+------------+------------------+--------------+-------------+----------+
|       3|2026-01-04|     S02|       P04|       4|     17000|           0|         Completed|         EMP04|        68000|    Medium|
|       9|2026-01-10|     S04|       P04|       2|     17000|           5|         Completed|         EMP10|        34000|     Small|
|      15|2026-01-16|     S02|       P04|       8|     17000|          15|         Completed|         EMP04|       136000|     Large|
|      21|2026-01-22|     S04|       P04|       6|     17000|           0|          Returned|         EMP10|       102000|    Medium|
|       2|2026-01-03|     S01|       P03|       3|      3500| 

**Part D | Window and date functions**

In [16]:
#13
from pyspark.sql.functions import dense_rank
from pyspark.sql.functions import window

joined = sales_df.join(
    products_df,
    on = 'product_id',
    how = 'inner'
)

products_revenue_df = joined.groupBy(
    'category',
    'product_id'
).agg(
    sum('gross_revenue').alias('total_revenue')
)

window_spec = Window.partitionBy('category').orderBy(
    col('total_revenue').desc()
)

result = products_revenue_df.withColumn(
    'dense_rank',
    dense_rank().over(window_spec)
)

result.show()


+-----------+----------+-------------+----------+
|   category|product_id|total_revenue|dense_rank|
+-----------+----------+-------------+----------+
|Accessories|       P03|    233334500|         1|
|Electronics|       P01|   4000080000|         1|
|Electronics|       P06|   1999920000|         2|
|Electronics|       P02|   1166704000|         3|
| Networking|       P05|    299992500|         1|
|     Office|       P04|   1416678000|         1|
+-----------+----------+-------------+----------+



In [17]:
#14
window_spec = Window.partitionBy('store_id').orderBy(
    'order_date','order_id'
).rowsBetween(
    Window.unboundedPreceding,
    Window.currentRow
)
sales_df = sales_df.withColumn(
    'running_total',
    sum('gross_revenue').over(window_spec)
)
sales_df.show()


+--------+----------+--------+----------+--------+----------+------------+------------------+--------------+-------------+----------+-------------+
|order_id|order_date|store_id|product_id|quantity|unit_price|discount_pct|transaction_status|salesperson_id|gross_revenue|order_size|running_total|
+--------+----------+--------+----------+--------+----------+------------+------------------+--------------+-------------+----------+-------------+
|     180|2026-01-01|     S01|       P01|       5|     60000|           0|         Completed|         EMP01|       300000|    Medium|       300000|
|     360|2026-01-01|     S01|       P01|       1|     60000|           0|         Completed|         EMP01|        60000|     Small|       360000|
|     540|2026-01-01|     S01|       P01|       5|     60000|           0|         Completed|         EMP01|       300000|    Medium|       660000|
|     720|2026-01-01|     S01|       P01|       1|     60000|           0|         Completed|         EMP01|    

In [18]:
#15
from pyspark.sql.functions import month
sales_df = sales_df.withColumn(
    'month',
    month(col('order_date'))
)
sales_df.filter(
    col('transaction_status') == 'Completed'
).groupby('month').agg(
    sum('gross_revenue').alias('Total gross revenue'
    )).orderBy('month').show()


+-----+-------------------+
|month|Total gross revenue|
+-----+-------------------+
|    1|         1278728500|
|    2|         1027585500|
|    3|         1212087500|
|    4|         1218921000|
|    5|         1277560500|
|    6|         1089027500|
+-----+-------------------+



**Part E | Lazy execution**

In [19]:
#16
filtering = sales_df.filter(
    col('transaction_status') == 'Completed'
)
print("Applied filters to get only completed orders")
add_column = filtering.withColumn(
    'discount_amount',
    col('gross_revenue') * (col('discount_pct') / 100)
)
print("Discount amount calculated and add a new column")
selection = add_column.select(
    'order_id',
    'product_id',
    'quantity',
    'unit_price',
    'discount_amount'
)
print("Applied select()")
#Action
selection.show()


Applied filters to get only completed orders
Discount amount calculated and add a new column
Applied select()
+--------+----------+--------+----------+---------------+
|order_id|product_id|quantity|unit_price|discount_amount|
+--------+----------+--------+----------+---------------+
|       3|       P04|       4|     17000|            0.0|
|       9|       P04|       2|     17000|         1700.0|
|      15|       P04|       8|     17000|        20400.0|
|       2|       P03|       3|      3500|            0.0|
|       8|       P03|       1|      3500|          175.0|
|      20|       P03|       5|      3500|            0.0|
|      26|       P03|       3|      3500|          525.0|
|       4|       P05|       5|      4500|            0.0|
|      10|       P05|       3|      4500|         1350.0|
|      16|       P05|       1|      4500|          675.0|
|       5|       P06|       6|     24000|         7200.0|
|      17|       P06|       2|     24000|         7200.0|
|      23|       P06

In [20]:
#17
aggregation = sales_df.groupBy('product_id').agg(
    sum('gross_revenue').alias('Total revenue')
)
aggregation.explain()
aggregation.collect()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[product_id#33], functions=[sum(gross_revenue#130L)])
   +- HashAggregate(keys=[product_id#33], functions=[partial_sum(gross_revenue#130L)])
      +- Project [product_id#33, (cast(quantity#35 as bigint) * unit_price#40L) AS gross_revenue#130L]
         +- SortMergeJoin [product_id#33], [product_id#0], LeftOuter
            :- Sort [product_id#33 ASC NULLS FIRST], false, 0
            :  +- Exchange hashpartitioning(product_id#33, 200), ENSURE_REQUIREMENTS, [plan_id=4762]
            :     +- Project [concat(P, lpad(cast(((id#30L % 6) + 1) as string), 2, 0)) AS product_id#33, cast(((id#30L % 8) + 1) as int) AS quantity#35]
            :        +- Range (1, 100001, step=1, splits=2)
            +- Sort [product_id#0 ASC NULLS FIRST], false, 0
               +- Exchange hashpartitioning(product_id#0, 200), ENSURE_REQUIREMENTS, [plan_id=4763]
                  +- Project [product_id#0, list_price#3L AS unit_price

[Row(product_id='P04', Total revenue=1416678000),
 Row(product_id='P03', Total revenue=233334500),
 Row(product_id='P05', Total revenue=299992500),
 Row(product_id='P06', Total revenue=1999920000),
 Row(product_id='P02', Total revenue=1166704000),
 Row(product_id='P01', Total revenue=4000080000)]

**Part F | Caching and persistence**

In [21]:
#18
import time
filtered_df = sales_df.filter(
    col('transaction_status') == 'Completed'
)
filtered_df.cache()

start_time = time.time()
count1 = filtered_df.count()
end_time = time.time()
print("Count 1:",count1)
print("Count 1 Execution time:",end_time - start_time)

start_time = time.time()
count2 = filtered_df.count()
end_time = time.time()
print("Count 2:",count2)
print("Count 2 Execution time:",end_time - start_time)




Count 1: 77923
Count 1 Execution time: 11.892956495285034
Count 2: 77923
Count 2 Execution time: 2.3995020389556885


In [22]:
#19
filtered_df = sales_df.filter(
    col('quantity') > 5
)
filtered_df.persist(StorageLevel.MEMORY_AND_DISK)

print("Count: ",filtered_df.count())
filtered_df.show()

filtered_df.unpersist()

Count:  37500
+--------+----------+--------+----------+--------+----------+------------+------------------+--------------+-------------+----------+-------------+-----+
|order_id|order_date|store_id|product_id|quantity|unit_price|discount_pct|transaction_status|salesperson_id|gross_revenue|order_size|running_total|month|
+--------+----------+--------+----------+--------+----------+------------+------------------+--------------+-------------+----------+-------------+-----+
|     183|2026-01-04|     S02|       P04|       8|     17000|           0|         Completed|         EMP04|       136000|     Large|       204000|    1|
|     543|2026-01-04|     S02|       P04|       8|     17000|           0|         Completed|         EMP04|       136000|     Large|       408000|    1|
|     903|2026-01-04|     S02|       P04|       8|     17000|           0|          Returned|         EMP04|       136000|     Large|       612000|    1|
|    1263|2026-01-04|     S02|       P04|       8|     17000| 

DataFrame[order_id: bigint, order_date: date, store_id: string, product_id: string, quantity: int, unit_price: bigint, discount_pct: int, transaction_status: string, salesperson_id: string, gross_revenue: bigint, order_size: string, running_total: bigint, month: int]

In [23]:
#20
aggregation = sales_df.groupBy('product_id').agg(
    sum('gross_revenue').alias('Total revenue')
)
aggregation.cache()

#action
aggregation.count()

aggregation.explain()

#release
aggregation.unpersist()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- InMemoryTableScan [product_id#33, Total revenue#2428L]
      +- InMemoryRelation [product_id#33, Total revenue#2428L], StorageLevel(disk, memory, deserialized, 1 replicas)
            +- AdaptiveSparkPlan isFinalPlan=true
               +- == Final Plan ==
                  ResultQueryStage 3
                  +- *(3) HashAggregate(keys=[product_id#33], functions=[sum(gross_revenue#130L)])
                     +- *(3) HashAggregate(keys=[product_id#33], functions=[partial_sum(gross_revenue#130L)])
                        +- *(3) Project [product_id#33, (cast(quantity#35 as bigint) * unit_price#40L) AS gross_revenue#130L]
                           +- *(3) BroadcastHashJoin [product_id#33], [product_id#0], LeftOuter, BuildRight, false
                              :- ShuffleQueryStage 0
                              :  +- Exchange hashpartitioning(product_id#33, 200), ENSURE_REQUIREMENTS, [plan_id=5603]
                         

DataFrame[product_id: string, Total revenue: bigint]

**Part G | Repartition and coalesce**

In [24]:
#21
print("Number of Existing partition count:",sales_df.rdd.getNumPartitions())

sales_df8 = sales_df.repartition(8)
print("Number of partition count now:",sales_df8.rdd.getNumPartitions())


Number of Existing partition count: 2
Number of partition count now: 8


In [25]:
from pyspark.sql.functions.builtin import spark_partition_id
#22
sales_df8.groupBy(spark_partition_id().alias('partition_id')).count().show()

+------------+-----+
|partition_id|count|
+------------+-----+
|           0|12500|
|           1|12500|
|           2|12500|
|           3|12500|
|           4|12500|
|           5|12500|
|           6|12500|
|           7|12500|
+------------+-----+



In [26]:
#23
sales_df6 = sales_df8.repartition(6,'store_id')
sales_df6.select(
    'store_id',
    spark_partition_id().alias('partition_id')
).groupBy('partition_id','store_id').count().orderBy(
    "partition_id", "store_id"
).show()

+------------+--------+-----+
|partition_id|store_id|count|
+------------+--------+-----+
|           1|     S03|24999|
|           3|     S02|25001|
|           4|     S01|25001|
|           5|     S04|24999|
+------------+--------+-----+



In [27]:
#24
sales_df12 = sales_df.repartition(12)
sales_df4_coalesce = sales_df12.coalesce(4)
sales_df4 = sales_df.repartition(4)

sales_df4.explain()
sales_df4_coalesce.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Exchange RoundRobinPartitioning(4), REPARTITION_BY_NUM, [plan_id=6863]
   +- Project [order_id#31L, order_date#32, store_id#34, product_id#33, quantity#35, unit_price#40L, discount_pct#36, transaction_status#165, salesperson_id#39, gross_revenue#130L, order_size#382, running_total#469L, month(order_date#32) AS month#509]
      +- Window [sum(gross_revenue#130L) windowspecdefinition(store_id#34, order_date#32 ASC NULLS FIRST, order_id#31L ASC NULLS FIRST, specifiedwindowframe(RowFrame, unboundedpreceding$(), currentrow$())) AS running_total#469L], [store_id#34], [order_date#32 ASC NULLS FIRST, order_id#31L ASC NULLS FIRST]
         +- Sort [store_id#34 ASC NULLS FIRST, order_date#32 ASC NULLS FIRST, order_id#31L ASC NULLS FIRST], false, 0
            +- Exchange hashpartitioning(store_id#34, 200), ENSURE_REQUIREMENTS, [plan_id=6859]
               +- Project [order_id#31L, order_date#32, store_id#34, product_id#33, quantity#35, 

In [28]:
#25
sales_df4 = sales_df.repartition(4)
sales_df4.write.csv('sales_data.csv',header=True)



**Final Challenge**

In [29]:
#1
from pyspark.sql.functions import rank
completed_df = sales_df.filter(
    col('transaction_status') == 'Completed'
)

#2
completed_df = completed_df.fillna(0,subset=['discount_pct'])

#3
joined = products_df.join(
    completed_df,
    on = 'product_id',
    how = 'inner'
).join(
    stores_df,
    on = 'store_id',
    how = 'inner'
)

#4
revenue_df = joined.withColumn(
    'gross_revenue',
    col('quantity') * col('unit_price')
).withColumn(
    'discount_amount',
    col('gross_revenue') * (col('discount_pct') / 100)
).withColumn(
    'discounted_revenue',
    col('gross_revenue') - (col('discount_amount'))
)
#5, 6
aggregated_df = revenue_df.groupBy(
    'region',
    'category'
).agg(
    sum("gross_revenue").alias("total_gross_revenue"),
    sum("discounted_revenue").alias("total_revenue"),
    count("order_id").alias("transaction_count")
)

#7
window_spec = Window.partitionBy('region').orderBy(
    col('total_revenue').desc()
)
final_df = aggregated_df.withColumn(
    'dense_rank',
    dense_rank().over(window_spec)
)

#8
final_df.cache()

#9
final_df.orderBy(
    'region',
    'dense_rank'
).show()


#10
final_df.repartition(2).write.mode('overwrite').parquet('final_df.parquet')

final_df.unpersist()


+------+-----------+-------------------+-------------+-----------------+----------+
|region|   category|total_gross_revenue|total_revenue|transaction_count|dense_rank|
+------+-----------+-------------------+-------------+-----------------+----------+
| North|Electronics|          934848000|   8.551824E8|             6492|         1|
| North|     Office|          441388000|    4.12114E8|             6492|         2|
| North| Networking|          146097000| 1.35026775E8|             6494|         3|
| South|Electronics|         2156236000|  2.0242794E9|            19484|         1|
| South|     Office|          662524000|   6.060058E8|             6495|         2|
| South|Accessories|          113631000| 1.05018025E8|             6494|         3|
| South| Networking|           87669000|  8.2684125E7|             6494|         4|
|  West|Electronics|         2493348000|  2.2992404E9|            12985|         1|
|  West|Accessories|           68169500|    6.42985E7|             6493|    

DataFrame[region: string, category: string, total_gross_revenue: bigint, total_revenue: double, transaction_count: bigint, dense_rank: int]